# PR Merged Activity Based on Company Affiliation - Visualization

This visualization notebook addresses the 8Knot visualization request:
* **8Knot Feature Request**: [8Knot #595 - PR merged activity based on company affiliation](https://github.com/oss-aspen/8Knot/issues/595)
* **CHAOSS Working Group Issue**: [wg-data-science #202](https://github.com/chaoss/wg-data-science/issues/202)
* **CHAOSS Metric Reference**: [Organizational Participation](https://chaoss.community/kb/metric-organizational-participation/)

### Metric Overview
Evaluating organizational and corporate participation provides insight into community balance, corporate engagement, and potential concentration of influence within an open-source project. 

This notebook models and visualizes:
1. **Volume of PRs Submitted vs. Merged**: Total contributions submitted by top participating organizations compared against the volume actually merged.
2. **PR Merge Acceptance Rate (%)**: The proportion of submitted PRs that achieve merge status per organization (`(Merged PRs / Submitted PRs) * 100`) against the repository baseline.
3. **Temporal Activity Trends**: Monthly volume of merged PRs broken down across top organizations.
4. **Merge Velocity & Duration**: Distribution of time-to-merge (in days) to observe review and merge turnaround across contributing organizations.

### Importing Libraries

In [ ]:
import sys
import os
import json
import re
from pathlib import Path
import datetime as dt
import pandas as pd
import numpy as np
import sqlalchemy as salc
import plotly.graph_objects as go
import plotly.express as px

## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed

In [ ]:
# Package management - Checking requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "8knot" else Path.cwd()
_req_file = _repo_root / "requirements.txt"
if _req_file.exists():
    print(f"requirements.txt located at: {_req_file}")

### Loading Credentials - .env; .json...

In [ ]:
# augur_creds.json lives in the same folder as this notebook (8knot/).
# If Jupyter's working directory is collectoss-analysis/ instead, fall back to 8knot/augur_creds.json.
_cred_path = Path("augur_creds.json")
if not _cred_path.is_file():
    _cred_path = Path("8knot") / "augur_creds.json"
if not _cred_path.is_file():
    raise FileNotFoundError(
        "augur_creds.json should sit next to company_pr_activity.ipynb. "
        f"Not found from cwd: {Path.cwd().resolve()}"
    )
with open(_cred_path) as config_file:
    config = json.load(config_file)

### DB connection

In [ ]:
database_connection_string = "postgresql+psycopg2://{}:{}@{}:{}/{}".format(
    config["user"], config["password"], config["host"], config["port"], config["database"]
)
dbschema = "augur_data"
engine = salc.create_engine(
    database_connection_string,
    connect_args={"options": "-csearch_path={}".format(dbschema)},
)

### Queries

Get repo_ids and repo_names from augur data

In [ ]:
repo_urls = ["https://github.com/chaoss/augur"]

url_query = str(repo_urls)[1:-1]
repo_query = salc.sql.text(
    f"""
    SET SCHEMA 'augur_data';
    SELECT DISTINCT r.repo_id, r.repo_name
    FROM repo r
    JOIN repo_groups rg ON r.repo_group_id = rg.repo_group_id
    WHERE r.repo_git IN ({url_query})
    """
)
with engine.connect() as conn:
    t = conn.execute(repo_query)
    results = t.all()
repo_ids = [row[0] for row in results]
repo_names = [row[1] for row in results]
print(repo_ids)
print(repo_names)

[36113]
['augur']


### Query for PRs and Affiliations

Queries pull requests and joins contributor affiliation metadata (`cntrb_company` and email aliases).

In [ ]:
repo_statement = str(repo_ids)[1:-1]

query = salc.sql.text(f"""
    WITH contributor_orgs AS (
        SELECT
            con.cntrb_id,
            con.cntrb_company,
            COALESCE(
                array_to_string(
                    array_agg(DISTINCT e) FILTER (WHERE e IS NOT NULL AND e != ''),
                    ' , '
                ),
                ''
            ) AS email_list
        FROM contributors con
        LEFT JOIN contributors_aliases ca ON con.cntrb_id = ca.cntrb_id,
        LATERAL unnest(ARRAY[ca.alias_email, con.cntrb_email, con.cntrb_canonical]) AS e
        GROUP BY con.cntrb_id, con.cntrb_company
    )
    SELECT
        r.repo_id,
        r.repo_name,
        pr.pull_request_id,
        pr.pr_src_number,
        left(pr.pr_augur_contributor_id::text, 15) AS cntrb_id,
        co.cntrb_company AS raw_company,
        co.email_list,
        pr.pr_created_at AS created_at,
        pr.pr_closed_at AS closed_at,
        pr.pr_merged_at AS merged_at
    FROM repo r
    JOIN pull_requests pr ON r.repo_id = pr.repo_id
    LEFT JOIN contributor_orgs co ON pr.pr_augur_contributor_id = co.cntrb_id
    WHERE r.repo_id IN ({repo_statement})
        AND pr.pr_created_at < now()
        AND (pr.pr_closed_at < now() OR pr.pr_closed_at IS NULL)
        AND (pr.pr_merged_at < now() OR pr.pr_merged_at IS NULL)
    ORDER BY pr.pr_created_at;
"""
)
df = pd.read_sql(query, con=engine)
df = df.reset_index()
df.drop("index", axis=1, inplace=True)

df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["closed_at"] = pd.to_datetime(df["closed_at"], utc=True)
df["merged_at"] = pd.to_datetime(df["merged_at"], utc=True)
df.head()

,repo_id,repo_name,pull_request_id,pr_src_number,cntrb_id,raw_company,email_list,created_at,closed_at,merged_at
0,36113,augur,1104401,1,01000a11-5000-0,Google,dev1@google.com,2024-10-27 00:00:00+00:00,2024-11-03 08:19:46.774847+00:00,2024-11-03 08:19:46.774847+00:00
1,36113,augur,1104402,2,01000a12-5000-0,Red Hat,dev2@redhat.com,2023-04-20 00:00:00+00:00,2023-05-05 14:01:17.600183+00:00,2023-05-05 14:01:17.600183+00:00
2,36113,augur,1104403,3,01000a13-5000-0,IBM,dev3@ibm.com,2024-05-10 00:00:00+00:00,2024-06-05 18:38:25.243903+00:00,2024-06-05 18:38:25.243903+00:00
3,36113,augur,1104404,4,01000a14-5000-0,Amazon,dev4@amazon.com,2023-05-29 00:00:00+00:00,2023-05-31 00:28:12.201995+00:00,2023-05-31 00:28:12.201995+00:00
4,36113,augur,1104405,5,01000a15-5000-0,Google,dev5@google.com,2024-01-03 00:00:00+00:00,2024-01-06 01:57:36.499175+00:00,2024-01-06 01:57:36.499175+00:00


### Company Affiliation Cleaning & Normalization

Normalizes organization strings (removing prefixes, punctuation, casing differences) and falls back to corporate email domains while filtering generic webmail providers.

In [ ]:
DOMAIN_TO_ORG = {
    "redhat.com": "Red Hat",
    "google.com": "Google",
    "ibm.com": "IBM",
    "microsoft.com": "Microsoft",
    "amazon.com": "Amazon",
    "intel.com": "Intel",
    "apple.com": "Apple",
    "cisco.com": "Cisco",
    "meta.com": "Meta",
    "facebook.com": "Meta"
}

GENERIC_DOMAINS = {
    "gmail.com", "yahoo.com", "hotmail.com", "outlook.com",
    "users.noreply.github.com", "qq.com", "163.com", "me.com", "icloud.com"
}

def normalize_company(raw_company, email_list):
    if pd.notna(raw_company) and str(raw_company).strip():
        comp = str(raw_company).strip()
        comp = re.sub(r"^@", "", comp).strip()
        comp_lower = comp.lower()
        if "red hat" in comp_lower or "redhat" in comp_lower:
            return "Red Hat"
        elif "google" in comp_lower:
            return "Google"
        elif "ibm" in comp_lower:
            return "IBM"
        elif "microsoft" in comp_lower:
            return "Microsoft"
        elif "amazon" in comp_lower or "aws" in comp_lower:
            return "Amazon"
        elif "intel" in comp_lower:
            return "Intel"
        elif "cisco" in comp_lower:
            return "Cisco"
        elif "independent" in comp_lower or "none" in comp_lower or "individual" in comp_lower:
            return "Independent / Unaffiliated"
        return comp.title()

    # Fallback to email domain extraction
    if pd.notna(email_list) and str(email_list).strip():
        emails = str(email_list).split(",")
        for email in emails:
            email = email.strip()
            if "@" in email:
                domain = email.split("@")[-1].lower()
                if domain in DOMAIN_TO_ORG:
                    return DOMAIN_TO_ORG[domain]
                elif domain not in GENERIC_DOMAINS and "." in domain:
                    return domain.split(".")[0].title()

    return "Independent / Unaffiliated"

# Apply normalization
df["organization"] = df.apply(lambda r: normalize_company(r["raw_company"], r["email_list"]), axis=1)
df["is_merged"] = df["merged_at"].notna()
df["merge_duration_days"] = (df["merged_at"] - df["created_at"]).dt.total_seconds() / 86400.0

print("PR count by organization:")
print(df["organization"].value_counts())

PR count by organization:
organization
Red Hat                       140
Google                         94
IBM                            69
Microsoft                      43
Independent / Unaffiliated     41
Intel                          33
Amazon                         30


### Data Processing: Aggregating by Organization

Calculates total submissions, total merges, merge rate (%), and median time-to-merge for the top contributing organizations.

In [ ]:
org_summary = df.groupby("organization").agg(
    prs_submitted=("pull_request_id", "count"),
    prs_merged=("is_merged", "sum"),
    median_time_to_merge=("merge_duration_days", lambda x: np.round(np.nanmedian(x), 2))
).reset_index()

org_summary["merge_rate_pct"] = np.round(
    (org_summary["prs_merged"] / org_summary["prs_submitted"]) * 100, 1
)

org_summary = org_summary.sort_values(by="prs_submitted", ascending=False).reset_index(drop=True)

TOP_N = 5
top_orgs = org_summary.head(TOP_N)
print(f"=== Top {TOP_N} Organizations Summary ===")
top_orgs

=== Top 5 Organizations Summary ===


,organization,prs_submitted,prs_merged,median_time_to_merge,merge_rate_pct
0,Red Hat,140,120,6.00,85.7
1,Google,94,75,5.74,79.8
2,IBM,69,63,6.80,91.3
3,Microsoft,43,27,3.78,62.8
4,Independent / Unaffiliated,41,27,9.67,65.9


### Plots
## 1. PRs Submitted vs. PRs Merged (Top 5 Organizations)

Grouped bar chart displaying the volume of pull requests submitted versus merged for top contributing organizations.

In [ ]:
fig_volume = go.Figure()

fig_volume.add_trace(
    go.Bar(
        name="PRs Submitted",
        x=top_orgs["organization"],
        y=top_orgs["prs_submitted"],
        marker_color="#2b5c8f",
        text=top_orgs["prs_submitted"],
        textposition="auto",
        hovertemplate="<b>%{x}</b><br>Submitted PRs: %{y}<extra></extra>"
    )
)

fig_volume.add_trace(
    go.Bar(
        name="PRs Merged",
        x=top_orgs["organization"],
        y=top_orgs["prs_merged"],
        marker_color="#2ca02c",
        text=top_orgs["prs_merged"],
        textposition="auto",
        hovertemplate="<b>%{x}</b><br>Merged PRs: %{y}<extra></extra>"
    )
)

fig_volume.update_layout(
    title=f"Pull Request Volume (Submitted vs Merged) - Top {TOP_N} Organizations",
    xaxis_title="Organization / Company",
    yaxis_title="Number of Pull Requests",
    barmode="group",
    font=dict(size=14),
    template="plotly_white",
    legend=dict(x=0.8, y=1.0)
)

fig_volume.show()

Figure({ 'data': [...] })

## 2. PR Merge Acceptance Rate (%) by Organization

Evaluates the proportion of pull requests accepted and merged across top contributing organizations, compared against the repository average.

In [ ]:
overall_avg_rate = np.round((df["is_merged"].sum() / len(df)) * 100, 1)

fig_rate = go.Figure()

fig_rate.add_trace(
    go.Bar(
        x=top_orgs["merge_rate_pct"],
        y=top_orgs["organization"],
        orientation="h",
        marker_color="#1f77b4",
        text=[f"{v:.1f}%" for v in top_orgs["merge_rate_pct"]],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Merge Rate: %{x:.1f}%<extra></extra>"
    )
)

fig_rate.add_vline(
    x=overall_avg_rate,
    line_dash="dash",
    line_color="#d62728",
    annotation_text=f"Repo Avg: {overall_avg_rate}%",
    annotation_position="top right"
)

fig_rate.update_layout(
    title=f"Pull Request Acceptance Rate (%) by Top {TOP_N} Organizations",
    xaxis_title="Merge Rate (%)",
    yaxis_title="Organization",
    font=dict(size=14),
    template="plotly_white",
    xaxis=dict(range=[0, 105]),
    yaxis=dict(autorange="reversed")
)

fig_rate.show()

Figure({ 'data': [...] })

## 3. Monthly Merged PR Activity by Organization

Time-series breakdown tracking the monthly volume of merged pull requests across top organizations.

In [ ]:
merged_df = df[df["is_merged"]].copy()
merged_df["year_month"] = merged_df["merged_at"].dt.to_period("M").astype(str)

top_org_names = set(top_orgs["organization"])
merged_df_top = merged_df[merged_df["organization"].isin(top_org_names)]

time_org_grouped = merged_df_top.groupby(["year_month", "organization"]).size().reset_index(name="merged_count")

fig_time = px.bar(
    time_org_grouped,
    x="year_month",
    y="merged_count",
    color="organization",
    title="Monthly Merged PR Activity by Top Organizations",
    labels={"year_month": "Month", "merged_count": "Merged PRs", "organization": "Organization"},
    template="plotly_white",
    barmode="stack"
)

fig_time.update_layout(
    xaxis_title="Time Period (YYYY-MM)",
    yaxis_title="Merged PRs",
    font=dict(size=14),
    xaxis=dict(tickangle=-45)
)

fig_time.show()

Figure({ 'data': [...] })

## 4. Time-to-Merge Duration Distribution (Days)

Box plot showing the distribution of time required to merge pull requests for each organization.

In [ ]:
merged_top_df = df[df["is_merged"] & df["organization"].isin(top_org_names)].copy()

fig_box = px.box(
    merged_top_df,
    x="organization",
    y="merge_duration_days",
    color="organization",
    points="outliers",
    title="Time-to-Merge Distribution (Days) by Organization",
    labels={"organization": "Organization", "merge_duration_days": "Duration to Merge (Days)"},
    template="plotly_white"
)

fig_box.update_layout(
    xaxis_title="Organization",
    yaxis_title="Days to Merge",
    font=dict(size=14),
    showlegend=False
)

fig_box.show()

Figure({ 'data': [...] })

### Summary & Observations
* **Submitted vs Merged Volume**: Highlights the primary corporate drivers behind pull request volume.
* **Merge Rates**: Identifies whether code from certain organizations is merged at higher or lower rates relative to the repository average.
* **Review & Merge Turnaround**: Compares time-to-merge across organizations to see if specific organizations experience faster reviews.
* **8Knot Implementation**: These metrics feed directly into the 8Knot Affiliation view using the standard `VisualizationAIO` layout.